# Dataset figures for the thesis

Generates the figures marked `\TODO{FIGURE n}` in the Datasets chapter.
All decoding uses the repo's own functions rather than local
reimplementations, so the glosses stay correct if an encoding changes.

Set the paths in the config cell, then run top to bottom. Figure 3 needs no
generated data and can be run on its own.

## What each figure is for

Not decoration — each carries an argument the prose makes poorly.

**Fig 1 (Sort-of-CLEVR scene + questions + answers).** Establishes the task and
shows the *arity gradient*: one object, two objects, three objects, all asked
about the same scene. Seeing all three against one image is what makes "the
families differ in how many objects must be considered jointly" concrete.
Also shows questions are structured vectors, not text.

**Fig 2 (SQOOP positive/negative pair).** Shows the hard-negative construction:
same question, same objects, opposite answers. The label turns on geometry, not
on which objects are present. Without this, "hard negative" is taken on trust.

**Fig 3 (SQOOP pairing matrix).** The load-bearing figure. Every object appears
many times on both axes; only *combinations* are withheld. That is the whole
justification for calling SQOOP a test of recombination rather than of
unfamiliar objects, and it makes `rhs_variety` legible as a continuous axis
(supporting "report a curve, not a point"). Hard to follow in prose.

**Fig 4 (coalitions timeline)** and **Fig 5 (graph families)** are not generated
here — see the last cell.

In [1]:
import sys
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ------------------------------------------------------------------ paths
REPO_ROOT = Path("~/syncnet").expanduser()                  # <- your repo root
SOC_NPZ   = r'/home/nik/workspace/ImperialWork/msc_project/SyncNetProject/data/sort_of_clevr-seed1-train36000-test1000-img75-obj5-q10-t-1/test.npz'      # <- adjust
SQOOP_NPZ = r'/home/nik/workspace/ImperialWork/msc_project/SyncNetProject/data/sqoop-rhs1/val_seen.npz'
FIGDIR    = Path("figures"); FIGDIR.mkdir(exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))

# ------------------------------------------------------------------ style
mpl.rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["DejaVu Serif"],
    "font.size":          9,
    "axes.labelsize":     9,
    "axes.titlesize":     9,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "figure.dpi":         140,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype":       42,   # embed TrueType; some checkers reject Type3
})

TEXTWIDTH_IN = 6.0   # \the\textwidth of the report class, in inches
                     # CHECK: \showthe\textwidth in LaTeX and update

def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGDIR / f"{name}.{ext}")
    print("wrote", FIGDIR / f"{name}.pdf")


# --------------------------------------------------- image display helper
# Sort-of-CLEVR draws with OpenCV and src/tasks/sort_of_clevr/data/constants.py
# stores COLOURS in **BGR** ('red' is (0, 0, 255)), so stored scenes are BGR and
# must have channels reversed for matplotlib or red and blue swap.
# SQOOP renders with Pillow and is already RGB.
#
# The generator writes a plain white background with no border, so CROP_PX=0 is
# correct for freshly generated data; the option exists for data padded
# elsewhere. The black frame is added here because white scenes otherwise bleed
# into the page.
def show_image(ax, img, bgr=False, crop=0, frame=True, lw=1.0):
    a = np.asarray(img)
    if crop:
        a = a[crop:-crop, crop:-crop]
    if bgr:
        a = a[..., ::-1]
    ax.imshow(a, interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(frame); s.set_linewidth(lw); s.set_color("black")
    return ax

CROP_PX = 0

## Figure 3 — SQOOP pairing matrix

Run this one first: it needs no generated data.

The construction is lifted from `prepare_sqoop` in
`src/tasks/sqoop/data/generator.py`:

```python
py_rng = random.Random(base_seed)
for i, x in enumerate(SHAPES):
    ys = py_rng.sample(SHAPES[:i] + SHAPES[i+1:], rhs)
```

so each left-hand object is paired with exactly `rhs` distinct right-hand
objects. Reproducing it here keeps the figure runnable without a built
dataset, but **duplicates repo logic** — if the split changes, this cell must
change with it. It is the only such duplication left in the notebook.

In [2]:
import random
from src.tasks.sqoop.data.constants import SHAPES, RELATIONS

print(f"{len(SHAPES)} shapes, {len(RELATIONS)} relations: {RELATIONS}")


def train_pairs(rhs, base_seed=0, shapes=SHAPES):
    """(lhs, rhs) pairs seen in training. Mirrors prepare_sqoop."""
    py_rng = random.Random(base_seed)
    pairs = set()
    for i, x in enumerate(shapes):
        for y in py_rng.sample(shapes[:i] + shapes[i + 1:], rhs):
            pairs.add((x, y))
    return pairs


def pair_matrix(rhs, base_seed=0, shapes=SHAPES):
    """|S| x |S|: 1 seen in training, 0 held out, nan on the diagonal."""
    idx = {s: i for i, s in enumerate(shapes)}
    M = np.zeros((len(shapes), len(shapes)))
    for x, y in train_pairs(rhs, base_seed, shapes):
        M[idx[x], idx[y]] = 1.0
    np.fill_diagonal(M, np.nan)      # x != y: no self-pairs exist
    return M


n_ordered = len(SHAPES) * (len(SHAPES) - 1)
for r in (1, 2, 4, 8, 18, 35):
    n = int(np.nansum(pair_matrix(r)))
    print(f"rhs={r:>2}: {n:>4} / {n_ordered} train pairs ({100*n/n_ordered:>5.1f}% seen)")

36 shapes, 4 relations: ['left_of', 'right_of', 'above', 'below']
rhs= 1:   36 / 1260 train pairs (  2.9% seen)
rhs= 2:   72 / 1260 train pairs (  5.7% seen)
rhs= 4:  144 / 1260 train pairs ( 11.4% seen)
rhs= 8:  288 / 1260 train pairs ( 22.9% seen)
rhs=18:  648 / 1260 train pairs ( 51.4% seen)
rhs=35: 1260 / 1260 train pairs (100.0% seen)


In [3]:
RHS_TO_SHOW = [1, 4, 18]   # low / mid / high; 35 is the IID control

fig, axes = plt.subplots(
    1, len(RHS_TO_SHOW),
    figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN / len(RHS_TO_SHOW) + 0.55))

cmap = mpl.colors.ListedColormap(["#e8e8e8", "#2b2b2b"])   # held out / seen
cmap.set_bad("#ffffff")                                     # diagonal

for ax, rhs in zip(axes, RHS_TO_SHOW):
    M = pair_matrix(rhs)
    ax.imshow(M, cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
    seen = 100 * np.nansum(M) / n_ordered
    # no LaTeX escaping: text.usetex is False, so "\%" would render literally
    ax.set_title(f"$\\mathrm{{rhs}}={rhs}$\n{seen:.0f}% of pairs seen", pad=6)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(0.5); s.set_color("#888")

axes[0].set_ylabel("left-hand object")
for ax in axes:
    ax.set_xlabel("right-hand object")

handles = [Rectangle((0, 0), 1, 1, fc="#2b2b2b"),
           Rectangle((0, 0), 1, 1, fc="#e8e8e8")]
fig.legend(handles, ["seen in training", "held out"],
           loc="lower center", ncol=2, frameon=False,
           bbox_to_anchor=(0.5, -0.06))

save(fig, "sqoop_pairing_matrix")
plt.show()

wrote figures/sqoop_pairing_matrix.pdf


/tmp/ipykernel_3577566/2960230039.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**For the caption:** each cell is one ordered object pair; the diagonal is
empty because a question never relates an object to itself. Every object appears
many times on both axes at every setting — only the *combinations* are withheld.
The seen fraction is exactly `rhs`/35, so the axis is linear in pair coverage
even though the plotted values are not evenly spaced.

## Figure 2 — SQOOP example scenes

Uses `decode_question` from `src/tasks/sqoop/data/constants.py`, which returns a
joined string such as `"A left_of B"`.

Picks one positive and one negative of the **same** question, which is what
makes the hard-negative construction visible: both scenes contain the queried
objects and differ only in geometry.

Note the negatives are more constrained than the positives — they additionally
require `x rel y'` and `x' rel y` to hold — so their distractor placement is
rejection-biased. That does not affect the figure, but it is why negative scenes
sometimes look more crowded toward one side.

In [4]:
from src.tasks.sqoop.data.constants import decode_question, encode_question

d = np.load(SQOOP_NPZ, allow_pickle=False)
images, questions, answers = d["images"], d["questions"], d["answers"]
print(f"{len(images)} examples, images {images.shape[1:]}, "
      f"positive rate {answers.mean():.4f}")
print("decode example:", decode_question(questions[0]))

# Round-trip check: encode(decode(q)) should recover q.
try:
    toks = decode_question(questions[0]).split()
    print("round-trip ok:",
          np.array_equal(np.asarray(encode_question(*toks)), questions[0]))
except Exception as e:
    print("round-trip check skipped:", type(e).__name__, e)

# Find questions appearing with BOTH labels.
from collections import defaultdict
by_q = defaultdict(dict)
for i, (q, a) in enumerate(zip(questions, answers)):
    by_q[tuple(q)].setdefault(int(a), i)
both = [q for q, v in by_q.items() if len(v) == 2]
print(f"{len(both)} distinct questions appear with both labels")

# Glyph confusability: 1/I and 0/O look near-identical at print size, so skip
# any question using them. Also prefer a pair that is not A/B, which reads as a
# placeholder rather than a real draw from a 36-object vocabulary.
AMBIGUOUS = {"1", "I", "0", "O"}
cands = [q for q in both
         if not (AMBIGUOUS & set(decode_question(np.array(q)).split()))]
print(f"{len(cands)} of those avoid confusable glyphs")

for k, q in enumerate(cands[:10]):
    print(f"  [{k}] {decode_question(np.array(q))}")

288 examples, images (64, 64, 3), positive rate 0.5000
decode example: Z right_of Y
round-trip ok: True
144 distinct questions appear with both labels
108 of those avoid confusable glyphs
  [0] Z right_of Y
  [1] Y above A
  [2] M right_of Z
  [3] 4 above 9
  [4] 7 right_of 6
  [5] N below 3
  [6] 9 left_of W
  [7] 3 right_of B
  [8] C left_of R
  [9] V below B


In [11]:
PICK = 0     # index into `cands` from the printout above

qsel = cands[PICK]
i_pos, i_neg = by_q[qsel][1], by_q[qsel][0]

lhs, rel, rhs_o = decode_question(np.array(qsel)).split()
qtext = f'Question: is {lhs} {rel.replace("_", " ")} {rhs_o}?'

# Set True to ring the two queried objects. The panels are superficially
# similar by design, so at print size a reader may not see which objects the
# question is about; the figure's job is to make the construction legible.
ANNOTATE = False

fig, axes = plt.subplots(1, 2, figsize=(TEXTWIDTH_IN * 0.62, TEXTWIDTH_IN * 0.36))
for ax, idx, lab in ((axes[0], i_pos, "yes"), (axes[1], i_neg, "no")):
    show_image(ax, images[idx], bgr=False, crop=CROP_PX)   # Pillow -> already RGB
    ax.set_title(f"answer: {lab}", pad=4)

fig.suptitle(f'{qtext}', y=1.06)
save(fig, "sqoop_examples")
plt.show()

wrote figures/sqoop_examples.pdf


/tmp/ipykernel_3577566/4258642852.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 1 — Sort-of-CLEVR scene with one question per family

Uses `translate_question` and `translate_answer` from
`src/tasks/sort_of_clevr/data/translate.py`.

**`translate_answer` must be passed the question.** Count questions
(`count_same_shape`, `count_in_box`, `count_obtuse`) store their answer as
`count + 4`, sharing indices 4–9 with the colour answers; the function switches
between `ANSWERS` and `COUNT_ANSWERS` on question type. Called without the
question, a count of 2 silently renders as `"blue"`.

Note that ternary is not uniformly a count family: `on_band` (subtype 1) returns
yes/no. The count subtypes are ternary 0 and 2, plus binary 2.

Questions are stored per family, already grouped by scene, so no index
arithmetic is needed.

In [6]:
from src.tasks.sort_of_clevr.data.translate import (
    translate_question, translate_answer,
)

soc = np.load(SOC_NPZ, allow_pickle=False)
print("files:", soc.files)

imgs = soc["images"]
FAMILIES = [
    ("Non-relational",     "nonrel_questions",  "nonrel_answers"),
    ("Binary relational",  "binary_questions",  "binary_answers"),
    ("Ternary relational", "ternary_questions", "ternary_answers"),
]
for lab, qk, ak in FAMILIES:
    print(f"{lab:<20} {soc[qk].shape}  answers {soc[ak].shape}")

SCENE = 0

# Print every question and answer for this scene, so the three used in the
# figure can be chosen by eye -- ideally ones whose answers are checkable
# against the rendered image, since that is the figure a reader studies most.
for lab, qk, ak in FAMILIES:
    print(f"\n--- {lab} ---")
    for j, (q, a) in enumerate(zip(soc[qk][SCENE], soc[ak][SCENE])):
        print(f"  [{j}] {translate_question(q)}  ->  {translate_answer(a, q)}")

files: ['images', 'ternary_questions', 'ternary_answers', 'binary_questions', 'binary_answers', 'nonrel_questions', 'nonrel_answers', 'object_positions', 'object_shapes']
Non-relational       (1000, 10, 18)  answers (1000, 10)
Binary relational    (1000, 10, 18)  answers (1000, 10)
Ternary relational   (1000, 10, 18)  answers (1000, 10)

--- Non-relational ---
  [0] Is the red object on the left side of the image?  ->  no
  [1] What shape is the grey object?  ->  rectangle
  [2] What shape is the yellow object?  ->  rectangle
  [3] What shape is the red object?  ->  rectangle
  [4] Is the red object on the left side of the image?  ->  no
  [5] What shape is the yellow object?  ->  rectangle
  [6] Is the orange object in the top half of the image?  ->  no
  [7] What shape is the yellow object?  ->  rectangle
  [8] Is the orange object in the top half of the image?  ->  no
  [9] Is the yellow object in the top half of the image?  ->  yes

--- Binary relational ---
  [0] What shape is the

In [9]:
import textwrap

# One question index per family, set from the printout above.
CHOSEN = {"Non-relational": 0, "Binary relational": 0, "Ternary relational": 1}
WRAP   = 52     # characters per line in the question column

rows = []
for lab, qk, ak in FAMILIES:
    j = CHOSEN[lab]
    q, a = soc[qk][SCENE][j], soc[ak][SCENE][j]
    rows.append((lab, translate_question(q), translate_answer(a, q)))

fig = plt.figure(figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * 0.40))
gs  = fig.add_gridspec(1, 2, width_ratios=[1, 1.85], wspace=0.10)

show_image(fig.add_subplot(gs[0]), imgs[SCENE], bgr=True, crop=CROP_PX)

ax2 = fig.add_subplot(gs[1]); ax2.axis("off")
ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)

y = 1.0
for lab, q, a in rows:
    ax2.text(0.0, y, lab, weight="bold", va="top", fontsize=8.5)
    y -= 0.115
    for line in textwrap.wrap(q, width=WRAP):
        ax2.text(0.0, y, line, va="top", fontsize=8.5)
        y -= 0.105
    ax2.text(0.0, y, f"answer: {a}", va="top", fontsize=8.5)
    y -= 0.175

save(fig, "soc_example")
plt.show()

wrote figures/soc_example.pdf


/tmp/ipykernel_3577566/2837408914.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Circles may look slightly blobby: `obj_size` is small relative to the image,
so `cv2.circle` has few pixels to work with. That is a property of the data, not
of the figure — rendering at a larger `img_size` for illustration would look
cleaner but would no longer show what the model actually sees.

## Figures 4 and 5 — not generated here

**Fig 4, coalitions timeline.** Streams over time with shaded episode bands and
target annotations showing the switch from own-next-token to modular sum. Needs
the coalitions generator.

**Fig 5, graph families and phase placement.** Three panels — clusterable,
non-clique-realisable, frustrated — each showing the target graph beside an
attempted placement on the circle, with the frustrated panel making visible why
no placement exists. This is the original contribution of the chapter and is far
more convincing seen than described. Probably **TikZ rather than matplotlib**:
three small circle diagrams that must stay legible at quarter page.